# NxM validation: held-out detector pulses

This notebook applies the regularized 12-channel fit to a raw series not retained in the saved PCA training sample. The ignored `data` symlink is required to reproduce the recorded results.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import hashlib
import os
from pathlib import Path
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np

from importlib import reload


REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'src'))
import nxm
from nxm import NxMReconstructor
import pycrp
reload(pycrp)
reload(nxm)


paths = {
    'templates': REPO_ROOT / 'data/pca/templates_pca_543.joblib',
    'csd': REPO_ROOT / 'data/final_psd_and_templates/csd_results.joblib',
    'training': REPO_ROOT / 'data/pca/traces_pca_543.joblib',
    'candidate': REPO_ROOT / 'data/trace_signal/trace_z1_23231216_233807.joblib',
}
DATA_AVAILABLE = all(path.exists() for path in paths.values())
if not DATA_AVAILABLE:
    missing = [str(path) for path in paths.values() if not path.exists()]
    print('Held-out validation skipped. Restore the ignored data symlink and these artifacts:')
    print('\n'.join(missing))

## Load artifacts and prove the holdout split

In [ ]:
if DATA_AVAILABLE:
    templates = joblib.load(paths['templates'], mmap_mode='r')
    csd_artifact = joblib.load(paths['csd'], mmap_mode='r')
    training = joblib.load(paths['training'], mmap_mode='r')
    candidate = joblib.load(paths['candidate'], mmap_mode='r')
    channels = list(csd_artifact['channels_ordered'])

    def fingerprint(trace):
        return hashlib.blake2b(np.asarray(trace).tobytes(), digest_size=16).digest()

    training_ids = {fingerprint(trace) for trace in training[channels[0]]}
    overlap = [index for index, trace in enumerate(candidate[channels[0]]) if fingerprint(trace) in training_ids]
    held_out_indices = [index for index in range(len(candidate[channels[0]])) if index not in overlap]
    print(f'Candidate events: {len(candidate[channels[0]])}')
    print(f'PCA-training overlap: {overlap}')
    print(f'Held-out events: {len(held_out_indices)}')
    assert overlap == [9]
    assert len(held_out_indices) == 26

## Fit all held-out events

The fit uses 4,000 low-frequency RFFT bins, the full 12-channel CSD, a ±150-sample initialization search, and a ±10-sample final window. Independent events are evaluated concurrently; the fit itself remains deterministic.

In [ ]:

frequency_limit = 10000
if DATA_AVAILABLE:
    means = np.stack([templates['template_avg'][channel] for channel in channels])
    components = np.stack([templates['template_pca'][channel] for channel in channels])
    variances = np.stack([templates['variance_pca'][channel] for channel in channels])
    # Legacy artifact stores one-sided spectral sums; convert to density at fs=1.
    csd = np.asarray(csd_artifact['csd_matrices']) / means.shape[1]
    
    noise_model = pycrp.qet.NoiseModel(n = 32768, fs = 1)
    noise_model.set_csd(csd.transpose(1, 2, 0))
    
    # Build NxMReconstructor for all channels, and for each channel individually (1xM)
    reconstructor = nxm.NxMReconstructor(
        means, components, variances, csd,
        channel_names=channels, 
        frequency_limit=frequency_limit,
    )
    mean_only = nxm.NxMReconstructor(
        means, np.empty((len(channels), 0, means.shape[1])),
        np.empty((len(channels), 0)), csd,
        channel_names=channels, 
        frequency_limit=frequency_limit,
    )
    reconstructor_singles = [nxm.NxMReconstructor(
        means[ich:ich+1], components[ich:ich+1], variances[ich:ich+1], csd[:, ich:ich+1, ich:ich+1],
        frequency_limit=frequency_limit
        ) for ich in range(len(channels))]
    
    # Build OF filters for each channel
    t0_mean = templates["metadata"]["template_t0"]
    OFs = [pycrp.filter.OF(means[i], csd[:,i,i].real, fs = 1, lowfreq_chi2_bins =100) for i in range(len(channels))]
    
    # Get data
    traces = []
    for event_index in held_out_indices:
        event = np.stack([np.asarray(candidate[channel][event_index], dtype=float) for channel in channels])
        event -= np.mean(event[:, :16000], axis=1, keepdims=True)
        traces.append(event)


In [ ]:
noise_traces, time_axis = noise_model.generate_noise_multichannel()


for t in noise_traces:
    plt.plot(t/32768**0.5)

In [ ]:
# Get one event

ievent = 25
event_raw = traces[ievent]

event = event_raw


# # rescale the event
# amp_new = 800
# noise_traces, time_axis = noise_model.generate_noise_multichannel()
# event = []
# for i, noise in enumerate(noise_traces):
#     event.append(event_raw[i] / max(event_raw[i]) * amp_new + noise)


In [ ]:
## 1. analytical fit to get starting time of each channel

# One fall time
result_total, results, model_total, models = pycrp.fit.fit_channels(event, csd, len_fit=1000, t0_guess=16030, tau_rise_guess=30, tau_fall_guess=100, t0_bound=50, x_scale=[20,1,2], nfall=1)
# Two fall times
# result_total, results, model_total, models = pycrp.fit.fit_channels(event, csd, len_fit=2050, t0_guess=16030, tau_rise_guess=20, tau_fall_guess=50, t0_bound=50, x_scale=[20,1,1, 1,0.1], nfall=2)

for ich, ch in enumerate(channels):
    print(f'Channel {ch}: t0 = {results[ich]["params"]["t0"]}, t0_mean = {t0_mean}, shift = {int(results[ich]["params"]["t0"]-t0_mean)}')

In [ ]:
## Run 1xM
init_delays = []
init_amplitudes = []
init_delays_nxm = []
init_amplitudes_nxm = []
fit_results_1xm = []
for ich, ch in enumerate(channels):
    delay = int(results[ich]["params"]["t0"]-t0_mean)
    trace_aligned = np.roll(event[ich], -delay)
    init_delays.append(delay)
    init_amplitudes.append(OFs[ich].of0(trace_aligned)[0])
    
    # Run 1xm once with initial parameters
    amplitude = init_amplitudes[ich]    
    profile, model_fft = reconstructor_singles[ich]._profile_fft(np.array([np.fft.rfft(trace_aligned)[:frequency_limit]]).T, np.array([amplitude]), np.array([0]))
    model_1xm_init = np.fft.irfft(model_fft.T[0])
    
    fit_result = reconstructor_singles[ich].fit(
            np.array(trace_aligned)[None,:], initial_amplitudes= np.array([amplitude]), initial_delays=np.array([0]),
            delay_half_width=20,
            optimizer_options={'xtol': 0.1, 'ftol': 1e-2, 'maxiter': 20, 'maxfev': 1400},
            method='powell'
        )    
    
    fit_results_1xm.append(fit_result)
    init_delays_nxm.append(delay + fit_result.delays[0])
    init_amplitudes_nxm.append(fit_result.amplitudes[0])


## Run NxM with initial parameters from 1xM
# Do not use the built-in initialization, as it is not guaranteed to be optimal for the NxM case
# amplitudes, delays = reconstructor.initialize(event, (-150, 150))
fit_result_nxm =  reconstructor.fit(
    event, initial_amplitudes=init_amplitudes_nxm, initial_delays=init_delays_nxm,
    delay_half_width=20,
    optimizer_options={'gtol': 1e-3, 'ftol': 1e-9, 'maxiter': 100, 'maxfun': 1400},
    fix_delay=True, method='powell'
)
model_nxm_fit = reconstructor.reconstruct(fit_result_nxm)



In [ ]:

for ich, ch in enumerate(channels):
    delay = int(results[ich]["params"]["t0"]-t0_mean)
    trace = event[ich]
    amplitude = init_amplitudes[ich]
    fit_result = fit_results_1xm[ich]
    model_1xm_fit = reconstructor_singles[ich].reconstruct(fit_result)
    
    # Print difference of t0 and amplitude between initial and fitted 1xM model
    chi2_1xm = reconstructor_singles[ich].reduced_chi_square(fit_results_1xm[ich])
    chi2_nxm = reconstructor.reduced_chi_square(fit_result_nxm)
    print(f'Chi2 1xM = {chi2_1xm}, Chi2 NxM = {chi2_nxm}')

    plt.plot(trace, label=f'Data')
    plt.plot(models[ich], label=f'Fit (1 exp fall time)')
    plt.plot(np.roll(means[ich]*amplitude, delay), label=f'Fit (OF0)')
    plt.plot(np.roll(model_1xm_fit[0], int(init_delays[ich])), label=f'Fit (1xM)')
    plt.plot(model_nxm_fit[ich], label=f'Fit (NxM)')

    plt.xlim(15900, 16800)
    plt.xlabel('Sample Index')
    plt.ylabel('Amplitude')
    plt.legend()
    plt.show()
    # break
    

In [ ]:
fit_result_nxm